# 602 Stride: source-input-fair direct-action LSTM sweep

Fairness contract: training and inference both receive exactly PC plus cache-line address, matching Stride's external inputs. The stateful LSTM learns an unbounded count and free-running autoregressive direct cache-line deltas. Normal actions are supervision only; no normal tracker, degree, threshold, page-offset table, request budget, future row, or semantic hand feature enters inference.

In [ ]:
import hashlib, os, pathlib, shutil, subprocess, sys, tarfile, torch
from google.colab import userdata
assert torch.cuda.is_available(), 'Select a GPU runtime (A100 preferred)'
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
REPO='/content/cache_arch'
TOKEN=userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add GITHUB_TOKEN to Colab Secrets'
ASKPASS='/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n')
os.chmod(ASKPASS,0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':ASKPASS,'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN':TOKEN})
try:
    if not os.path.isdir(REPO): subprocess.run(['git','clone','https://github.com/Angelawoo572/cache_arch.git',REPO],check=True,env=env)
    else: subprocess.run(['git','-C',REPO,'pull','--ff-only','origin','main'],check=True,env=env)
finally: pathlib.Path(ASKPASS).unlink(missing_ok=True)
print(torch.cuda.get_device_name(0), subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
RUN_ID='602_offline_lstm_stride_variable_delta_free_running_v7_seed7'
DRIVE_ROOT=f'/content/drive/MyDrive/cache_prefetch_602_stride/{RUN_ID}'
INPUT_DIR=f'{DRIVE_ROOT}/colab_input'; OUTPUT_ROOT=f'{DRIVE_ROOT}/colab_output'
os.makedirs(DRIVE_ROOT,exist_ok=True)
archive_name=f'{RUN_ID}.colab_input.tar.gz'
uploaded=files.upload()
assert archive_name in uploaded, f'Select {archive_name}'
archive=f'{DRIVE_ROOT}/{archive_name}'; pathlib.Path(archive).write_bytes(uploaded[archive_name])
if os.path.isdir(INPUT_DIR): shutil.rmtree(INPUT_DIR)
os.makedirs(INPUT_DIR,exist_ok=True)
with tarfile.open(archive,'r:gz') as handle: handle.extractall(INPUT_DIR)
for record in pathlib.Path(f'{INPUT_DIR}/SHA256SUMS').read_text().splitlines():
    expected,name=record.split(maxsplit=1); name=name.lstrip('*')
    assert hashlib.sha256(pathlib.Path(f'{INPUT_DIR}/{name}').read_bytes()).hexdigest()==expected
print('verified',archive)

In [ ]:
import gzip, json
TRACE='602.gcc_s-734B'; POLICY='stride'; ROLES=['train','eval']
STREAMS={role:f'{INPUT_DIR}/{TRACE}.{role}_stream.csv.gz' for role in ROLES}
for role,path in STREAMS.items():
    assert os.path.isfile(path),path
    with gzip.open(path,'rb') as handle:
        digest=hashlib.sha256()
        for block in iter(lambda:handle.read(1024*1024),b''): digest.update(block)
    print(role,digest.hexdigest())
SCRIPT=f'{REPO}/formal_NN_training/experiments/602_offline_lstm_stride/python/train_and_offline_infer.py'
assert os.path.isfile(SCRIPT),SCRIPT

In [ ]:
LOCAL_OUTPUT=f'/content/{RUN_ID}_colab_output'
if os.path.isdir(LOCAL_OUTPUT): shutil.rmtree(LOCAL_OUTPUT)
os.makedirs(LOCAL_OUTPUT)
HIDDEN_SIZES=[8,16,32,64,128]
SWEEP=[]
for hidden in HIDDEN_SIZES:
    tag=f'h{hidden}'; out=f'{LOCAL_OUTPUT}/{tag}'
    cmd=[sys.executable,SCRIPT,'--train-stream',STREAMS['train'],'--eval-stream',STREAMS['eval'],'--out-dir',out,'--device','cuda','--seed','7','--epochs','8','--chunk-len','1024','--batch-chunks','32','--hidden-size',str(hidden)]
    print('\nTraining',tag,' '.join(cmd),flush=True)
    subprocess.run(cmd,check=True)
    meta=json.loads(pathlib.Path(f'{out}/run_metadata.json').read_text())
    expected={'experiment_revision':'source_input_variable_delta_free_running_v7','neural_role':'standalone_direct_action_prefetcher','same_external_input_contract':True,'training_inference_input_encoder_identical':True,'decoder_training_mode':'free_running_autoregressive_same_as_inference','decoder_previous_teacher_action_used_as_input':False,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'normal_policy_request_rate_used_as_budget':False,'normal_policy_constants_used_by_neural_inference':False,'probability_threshold_used':False,'neural_degree_cap':None,'fixed_page_offset_classes':None,'same_page_rule_used_by_neural_inference':False,'future_label_window_used':False,'handcrafted_semantic_features_used':False,'manual_loss_weights_used':False,'training_regularization_used':False,'inference_policy_hardcodes_used':False,'threshold_related_hardcodes_used':False,'learned_request_count':True,'nn_generates_own_target_addresses':True,'matched_normal_prefetcher':POLICY}
    bad={k:(meta.get(k),v) for k,v in expected.items() if meta.get(k)!=v}
    assert not bad,bad
    assert meta['training_runtime_fields']==['pc','cache_line_address']==meta['inference_runtime_fields']
    encoder_hashes={meta.get('runtime_encoder_sha256'),meta.get('training_runtime_encoder_sha256'),meta.get('inference_runtime_encoder_sha256')}
    assert len(encoder_hashes)==1 and isinstance(next(iter(encoder_hashes)),str) and len(next(iter(encoder_hashes)))==64,encoder_hashes
    assert meta.get('decoder_free_running_self_test')=='PASS',meta.get('decoder_free_running_self_test')
    SWEEP.append({k:meta[k] for k in ('hidden_size','parameter_count','decision_rule','offline_stride_entries','offline_lstm_entries','heldout_behavior_metrics')})
if os.path.isdir(OUTPUT_ROOT): shutil.rmtree(OUTPUT_ROOT)
shutil.copytree(LOCAL_OUTPUT,OUTPUT_ROOT)
pathlib.Path(f'{OUTPUT_ROOT}/sweep_manifest.json').write_text(json.dumps({'trace':TRACE,'policy':POLICY,'revision':'source_input_variable_delta_free_running_v7','points':SWEEP},indent=2)+'\n')
print(json.dumps(SWEEP,indent=2))

In [ ]:
for hidden in HIDDEN_SIZES:
    out=f'{OUTPUT_ROOT}/h{hidden}'
    assert all(os.path.isfile(f'{out}/{name}') for name in ('offline_stride.replay.csv','offline_lstm.replay.csv','model.pt','run_metadata.json','training_history.csv'))
OUTPUT_ARCHIVE=f'{DRIVE_ROOT}/{RUN_ID}.colab_output.tar.gz'
with tarfile.open(OUTPUT_ARCHIVE,'w:gz') as archive:
    for item in pathlib.Path(OUTPUT_ROOT).iterdir(): archive.add(item,arcname=item.name)
print('DONE',OUTPUT_ARCHIVE,os.path.getsize(OUTPUT_ARCHIVE),'bytes')

Upload the output archive to the matching server run directory, then launch replay. This notebook never changes or recollects the input streams.
